# CS781 Phase 4 (CJPE) — v3: the REQUIRED techniques (full, partial, LoRA, LoRMA, IA3, knowledge distillation) across all four Phase 1 architectures, with a rule-checked selection of the six submissions

**What was missing from the earlier notebooks** (an outside review of `cs781-gpt2-pred-phase4.ipynb` against the Capstone Document, as you pasted it): only two GPT-2 experiments, no LoRMA, no distillation, only one architecture. I have not seen the PDF itself, so I treated that list as the rules, and **cell 9 enforces them**: six experiments, all four Phase 1 architectures, compulsory LoRMA, at least one distillation run, at least five different technique families. If the PDF says something different, edit `FAMILY` / `REQ_MODELS` at the top of cell 9.
**The old 3,000-step GPT-2 is not used anywhere** (the model loader ignores it).

## The experiments (cell 8) and what each one covers
| run | Phase 1 model | technique | why |
|---|---|---|---|
| **P1** | GPT-2 (decoder) | full fine-tuning | baseline recipe; saved as a distillation teacher |
| **P2** | encoder (ModernBERT-style) | full fine-tuning | expected best single model; saved as a distillation teacher |
| **P3** | encoder | **LoRMA+** (compulsory) | multiplicative low-rank adaptation of q and v: h' = (I + (alpha/r) B A) h, backbone frozen |
| **P4** | T5 encoder | LoRA | additive low-rank adaptation of q and v |
| **P5** | BiLSTM | partial (layer-wise) fine-tuning | embeddings and the lowest LSTM layer frozen. The BiLSTM has no attention layers, so adapters do not apply to it |
| **P6** (optional) | GPT-2 | IA3 | learned rescaling of keys, values and FFN activations |
| **P7** | half-depth GPT-2 (first 6 layers of the Phase 1 model) | **knowledge distillation** | student of the P1 + P2 ensemble (teacher margins on every training case); loss = alpha * CE + (1 - alpha) * T^2 * KL |
| **P7c** (control) | same student | none | so the benefit of distillation is measured; never one of the six |

**Models that are not ready yet are simply skipped.** Cell 8 starts with a table of which Phase 1 models are available and which experiments will run; every experiment cell checks its own model (re-scanning `/kaggle/input`, so a dataset attached mid-session is found) and skips only itself with a message if the model is missing. The distillation run uses whichever teachers exist (P1 and/or P2). Cell 9 then selects the best set of up to six runs: a fully rule-compliant six whenever one exists, otherwise the set that satisfies the most rules, with a clear status (for example 'architectures not covered yet: [bilstm]') and a `prediction.json` made from what exists. When the missing model is ready, run its cell and re-run cell 9.

**Minimum valid six:** P1, P2, P3, P4, P5, P7. P6 and the controls are optional extras (cell 9 picks the best valid set from everything in the ledger, and builds an ensemble of all non-control runs).
Every run uses the same training recipe as the v2 notebook (3-4 epochs, EMA weights, multi-window training from the last 2,048 tokens of each case, all windows averaged at inference, rate-matched threshold, test labels ignored), so differences between runs are due to the model and the technique.

## Run order and time (rough estimates from the first runs; 'pilot' = a 10-minute learning-rate search per technique, skippable by using the defaults)
setup -> CREDENTIALS check (cell 2: reads HF_TOKEN, KAGGLE_USERNAME and KAGGLE_KEY from Kaggle Secrets, verifies them first and stops early with the exact fix; nothing in this notebook asks you to type a token, so it also works in Save & Run All; the Hugging Face token is not needed if the CJPE data is attached) -> data -> models -> adapters -> trainer -> smoke test -> **P1 (80 min)** -> **P2 (30+70)** -> **P3 (30+60)** -> P4 (30+60) -> P5 (30+60) -> **P7 (teacher 30 + student 50)** -> final selection -> diagnostics. Optional: P6 (30+60), P7c (50).
P2 needs the pre-trained encoder dataset, P4 the T5 dataset and P5 the BiLSTM dataset attached (any dataset or notebook output holding their `final_model.pt`; they are found automatically). P7 needs the saved models of P1 and P2.

## What was tested (CPU, tiny models, synthetic data in the real file layout) and what was not
Every adapter on every architecture starts as the exact identity of the base model, trains only its own parameters (frozen weights verified unchanged), and partial freezing / layer truncation work; the BiLSTM cleanly rejects adapter methods. **Distillation is verified to drive learning**: a student trained against a deliberately wrong teacher with no hard labels scored AUC 0.000, with hard labels only 1.000. A saved model reloads to the exact recorded dev predictions (full fine-tuning and LoRMA). The selection cell accepts a valid set and refuses when LoRMA, distillation, an architecture, or five technique families is missing.
**Not tested:** the real CJPE files, GPU speed and memory, and real-model quality: the learning rates for the adapter methods are starting points (use the pilots), and expect modest gains over the GPT-2 baseline, not a jump to the top of the leaderboard.


In [ ]:
# ================= CELL 1 — setup =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path("/kaggle/working/cjpe"); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK (run it FIRST; it never asks you to type anything): reads HF_TOKEN, KAGGLE_USERNAME and KAGGLE_KEY from Kaggle Secrets, verifies them NOW, and stops early with the exact fix =====   [CELL 2]
# Needs: Internet ON, and the three Secrets added under Add-ons > Secrets with the 'Attached' toggle ON for THIS notebook (a committed run only sees attached secrets).
import os, glob, subprocess, shutil
from pathlib import Path
REQUIRE_HF, REQUIRE_KAGGLE = True, True        # True = stop now if that credential does not work; False = go on without it (no Hugging Face download / no backup upload)
HF_REPO = "Exploration-Lab/CS-781-Capstone"
def get_secret(name):
    """environment variable first, then the Kaggle Secret of the same name; quotes and spaces stripped; never printed"""
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
# ---- 1) Hugging Face token: only needed when the CJPE json files are not already on disk or attached as a dataset ----
have_local = all(glob.glob(f"/kaggle/input/**/{n}", recursive=True) or glob.glob(f"/kaggle/working/**/{n}", recursive=True) for n in ("train.json", "dev.json", "test.json"))
hf = get_secret("HF_TOKEN"); HF_OK = False
if have_local: report.append("Hugging Face token : not needed (the CJPE files are already on disk / attached)"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING (no HF_TOKEN secret or environment variable)"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or attach the CJPE files as a Kaggle dataset")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_REPO, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_REPO} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it: it is wrong, expired or revoked (create a new classic 'Read' token and update the Secret)" if code == 401 else "the token is valid but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
# ---- 2) Kaggle username + key: needed to upload the backup dataset ----
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append(f"Kaggle credentials  : MISSING ({'KAGGLE_USERNAME' if not ku else ''}{' and ' if not (ku or kk) else ''}{'KAGGLE_KEY' if not kk else ''} not found)"); problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing: add them as Secrets (the LEGACY API key from kaggle.com/settings > API)")
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed in this image)"); problems.append("the kaggle command is not installed, so the backup upload cannot run")
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}', the API accepted the key)")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); problems.append("the Kaggle username/key were refused: check that KAGGLE_KEY is the LEGACY API key and that Internet is ON")
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})"); problems.append(f"the Kaggle credentials could not be checked ({type(e).__name__}): is Internet ON?")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s) and switch 'Attached' ON for this notebook, then run again. (Set REQUIRE_HF / REQUIRE_KAGGLE = False at the top of this cell to continue without them.)")
if problems: print("\nContinuing without:", "; ".join(problems))
print("\nCredentials ready." if not problems else "")


In [ ]:
# ================= CELL 3 — CJPE data: locate (or download), detect the schema, tokenise ONLY what is needed (the end of each case), cache =================
HF_REPO = "Exploration-Lab/CS-781-Capstone"          # gated: Phase_4_CJPE/Train/train.json (737 MB), Train/dev.json (80 MB), Test/test.json (35 MB)
TAIL_MAX = 4096                                      # tokens kept from the END of every case (the end of a judgment is the most informative part)
TEXT_KEYS = ("text", "document", "case_text", "content", "judgment", "doc", "full_text", "case"); LABEL_KEYS = ("label", "labels", "decision", "outcome", "y", "gold", "judgement", "verdict")
ID_KEYS = ("id", "case_id", "doc_id", "document_id", "name", "file", "filename", "index")

def locate():
    got = {}
    for sp in ("train", "dev", "test"):
        c = [p for root in SEARCH_ROOTS for p in glob.glob(f"{root}/**/{sp}.json", recursive=True) if re.search(r"cjpe|phase[_ -]?4", p, re.I)]
        if c: got[sp] = max(c, key=os.path.getsize)
    return got
paths = locate()
if len(paths) < 3:
    print("CJPE files not found locally -> downloading from Hugging Face (accept the dataset conditions on its page first)")
    from huggingface_hub import HfApi, snapshot_download
    def hf_token_ok(tk):
        """True only if Hugging Face accepts the token AND the gated dataset is readable with it (a wrong token used to fail later with a confusing 401)"""
        try:
            api = HfApi(); me = api.whoami(token=tk); api.auth_check(HF_REPO, repo_type="dataset", token=tk); print(f"   accepted: user '{me.get('name')}', access to {HF_REPO} OK"); return True
        except Exception as e:
            code = getattr(getattr(e, "response", None), "status_code", None); print(f"   REFUSED (HTTP {code}, {type(e).__name__}): " + ("Hugging Face does not recognise this token" if code == 401 else "the token is valid but this account has no approved access to the dataset"
                  if code == 403 else str(e)[:120])); return False
    cands = []
    if os.environ.get("HF_TOKEN"): cands.append(("environment variable HF_TOKEN", os.environ["HF_TOKEN"]))
    try:
        from kaggle_secrets import UserSecretsClient; cands.append(("Kaggle secret HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN")))
    except Exception: pass
    token = None
    for src_, tk in cands:
        tk = (tk or "").strip().strip("'\""); print(f"trying the token from the {src_} ({len(tk)} characters)")
        if tk and hf_token_ok(tk): token = tk; break
        print(f"   -> not usable; if this is a Kaggle secret it is probably stale: update or delete it")
    assert token, "no working Hugging Face token and the CJPE files are not attached. This cell never asks you to type anything. Fix: add a Kaggle Secret named HF_TOKEN (Add-ons > Secrets, Attached ON for this notebook) holding a token of the account that was approved for the dataset, or attach the CJPE files as a Kaggle dataset"
    os.environ["HF_TOKEN"] = token
    local = snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["Phase_4_CJPE/*"], token=token, local_dir=str(WORK / "hf")); SEARCH_ROOTS.append(local); paths = locate()
assert len(paths) == 3, f"need train/dev/test json, found {paths}"
print({k: f"{os.path.getsize(v)/2**20:.0f} MiB  {v}" for k, v in paths.items()})

def normalise(obj):
    if isinstance(obj, dict):
        for wk in ("data", "documents", "examples", "cases", "items", "train", "dev", "test", "validation"):
            if wk in obj and isinstance(obj[wk], (list, dict)) and len(obj) <= 4: return normalise(obj[wk])
        out = []
        for k, v in obj.items():
            if isinstance(v, dict): r = dict(v); r.setdefault("id", k)
            elif isinstance(v, str): r = {"id": k, "text": v}
            else: continue
            out.append(r)
        return out
    return [r for r in obj if isinstance(r, dict)]
def detect(recs):
    keys = list(recs[0].keys()); strlen = {k: np.mean([len(str(r.get(k, ""))) for r in recs[:200]]) for k in keys if isinstance(recs[0].get(k), str)}
    tk = next((k for k in TEXT_KEYS if k in keys), None) or (max(strlen, key=strlen.get) if strlen else None); lk = next((k for k in LABEL_KEYS if k in keys), None)
    if lk is None:
        for k in keys:
            if k == tk: continue
            vals = {str(r.get(k)) for r in recs[:2000] if r.get(k) is not None}
            if 1 < len(vals) <= 5 and k not in ID_KEYS: lk = k; break
    return tk, lk, next((k for k in ID_KEYS if k in keys), None)
def binary_map(vals):
    s = {str(v).strip().lower() for v in vals if v is not None}
    if s <= {"0", "1", "0.0", "1.0", "true", "false"}: return lambda v: int(str(v).strip().lower() in ("1", "1.0", "true"))
    raise ValueError(f"label values {sorted(s)[:10]} are not 0/1: tell me what they mean (for example accepted/rejected) before training")

from tokenizers import Tokenizer
tpaths = sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True)) or sorted(glob.glob("/kaggle/working/**/legal_bpe_32k*.json", recursive=True))
assert tpaths, "attach the tokenizer file legal_bpe_32k_mask.json (it is inside your Phase 1 dataset)"
tok = Tokenizer.from_file(tpaths[0]); EOS, BOS, PAD = tok.token_to_id("[EOS]"), tok.token_to_id("[BOS]"), tok.token_to_id("[PAD]"); print("tokenizer:", tpaths[0], "| vocabulary", tok.get_vocab_size())

def prepare(split, path, bs=256):
    cache = WORK / "cache" / f"{split}.pkl"
    if cache.exists():
        d = pickle.load(open(cache, "rb")); print(f"[{split}] loaded cache: {len(d['tails'])} cases"); return d
    print(f"[{split}] reading {path} ..."); obj = json.load(open(path, encoding="utf-8")); recs = normalise(obj); del obj; gc.collect()
    tk, lk, ik = detect(recs); lmap = binary_map([r.get(lk) for r in recs[:5000]]) if lk else None
    print(f"[{split}] {len(recs)} cases | text field '{tk}' | label field '{lk}' | id field '{ik}'")
    tails, lens, labels, rids = [], [], [], []
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        for e in tok.encode_batch([str(r.get(tk, "")) for r in chunk], add_special_tokens=False):
            lens.append(len(e.ids)); tails.append(np.asarray((e.ids + [EOS])[-TAIL_MAX:], dtype=np.int32))
        labels += [lmap(r.get(lk)) if lmap else -1 for r in chunk]; rids += [r.get(ik, i + j) if ik else i + j for j, r in enumerate(chunk)]
        for r in chunk: r.clear()
    if split == "test":
        if lk and any(l >= 0 for l in labels): print("[test] NOTE: test.json contains a label field. The task says test labels are private, so they are IGNORED here and never used for anything.")
        labels = [-1] * len(labels)
    d = dict(tails=tails, lens=np.asarray(lens), labels=np.asarray(labels), rids=rids, keys=dict(text=tk, label=lk, id=ik)); pickle.dump(d, open(cache, "wb"), protocol=4); del recs; gc.collect(); return d
DATA = {sp: prepare(sp, p) for sp, p in paths.items()}
DATA["test"]["labels"] = np.full(len(DATA["test"]["tails"]), -1)       # test labels are private: blank them even if a cache from an earlier run held them
for sp, d in DATA.items():
    y = d["labels"]; print(f"[{sp}] cases {len(d['tails']):6d} | tokens p50/p90/p99: {int(np.percentile(d['lens'],50))}/{int(np.percentile(d['lens'],90))}/{int(np.percentile(d['lens'],99))} | label counts: {dict(Counter(y.tolist())) if (y>=0).any() else 'none (test)'}")
assert len(set(map(str, DATA["test"]["rids"]))) == len(DATA["test"]["rids"]), "duplicate test ids"
ytr = DATA["train"]["labels"]; print(f"majority-class baseline macro-F1 on dev would be about {0.5 * (2*max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)) / (1+max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)))):.3f}")


In [ ]:
# ================= CELL 4 — load Phase 1 models (all four architectures + the old GPT-2), document classifier, chunked batching =================
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
assert al_paths, "attach the dataset/notebook output that contains arch_loaders.py (written by cell 8 of the pre-training notebooks)"
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt": continue                # only the NEW Phase 1 models: the old 3,000-step checkpoint is deliberately never used
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family")
        if fam is None: continue                              # a checkpoint without a family entry is an old-format model: skipped
        key = "gpt2_new" if fam == "gpt2" else fam
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS})
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

class DocClassifier(nn.Module):
    """Encodes K chunks of a case (the last K*L tokens, right-aligned) with a Phase 1 backbone, pools each chunk, combines the chunks with attention pooling
       + the final chunk, and classifies. K=1 is the plain 'last L tokens' model. Chunks that are empty (short cases) are skipped, never fed to the backbone."""
    def __init__(self, backbone, hidden, causal, K, dropout=0.1, pool="last"):
        super().__init__(); self.backbone, self.hidden, self.causal, self.K, self.pool = backbone, hidden, causal, K, pool
        self.chunk_pos = nn.Parameter(torch.zeros(K, hidden)); self.proj = nn.Linear(hidden, hidden); self.att = nn.Linear(hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2))
    def encode_chunks(self, ids, mask):
        B, K, L = ids.shape; valid = mask.sum(-1) > 0; fi, fm = ids[valid], mask[valid]; T = int(fm.sum(1).max()); fi, fm = fi[:, :T], fm[:, :T]
        h = self.backbone(input_ids=fi, attention_mask=fm).last_hidden_state
        mm = fm.unsqueeze(-1).to(h.dtype); mean = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
        if self.causal:
            last = h[torch.arange(h.shape[0], device=h.device), fm.sum(1) - 1]; pooled = last if self.pool == "last" else 0.5 * (last + mean)      # pool='meanlast': average of the last-token and mean-pooled vectors
        else: pooled = mean
        out = pooled.new_zeros(B, K, self.hidden); out[valid] = pooled; return out, valid
    def forward(self, ids, mask):
        x, valid = self.encode_chunks(ids, mask); x = x + self.chunk_pos.to(x.dtype)
        s = self.att(torch.tanh(self.proj(x))).squeeze(-1).float().masked_fill(~valid, -1e4); w = torch.softmax(s, -1).to(x.dtype)
        return self.head(torch.cat([(w.unsqueeze(-1) * x).sum(1), x[:, -1]], -1))

def make_batch(tails, idx, K, L):
    """last K*L tokens of each case, split RIGHT-aligned into K chunks of <= L tokens (chunk K-1 is the very end); right-padded; returns ids/mask [B,K,L]"""
    B = len(idx); ids = np.full((B, K, L), PAD, dtype=np.int64); mask = np.zeros((B, K, L), dtype=np.int64)
    for b, i in enumerate(idx):
        t = tails[i][-K * L:]; n = len(t)
        for k in range(K - 1, -1, -1):
            end = n - (K - 1 - k) * L
            if end <= 0: break
            seg = t[max(0, end - L):end]; ids[b, k, :len(seg)] = seg; mask[b, k, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)

def set_backbone_dropout(backbone, p):
    """turn on dropout inside the backbone for fine-tuning (works where the model has nn.Dropout modules, e.g. GPT-2 and T5; a no-op for the others). Returns how many modules were changed."""
    n = 0
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p; n += 1
    return n

# ---- windows: window k = the k-th block of L tokens counted back from the end of the case (k=0 is the very end) ----
def n_windows(tail_len, L, W): return max(1, min(W, -(-tail_len // L)))
def get_window(tail, k, L):
    n = len(tail); end = n - k * L
    return tail[-L:] if end <= 0 else tail[max(0, end - L):end]
def make_window_batch(tails, idx, ks, L):
    B = len(idx); ids = np.full((B, 1, L), PAD, dtype=np.int64); mask = np.zeros((B, 1, L), dtype=np.int64)
    for b, (i, k) in enumerate(zip(idx, ks)):
        seg = get_window(tails[i], int(k), L); ids[b, 0, :len(seg)] = seg; mask[b, 0, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)

def truncate_layers(bb, n):
    """keep only the first n transformer layers (initialises a smaller student from the same Phase 1 weights); implemented for the GPT-2 family"""
    assert bb.family == "gpt2", "layer truncation is implemented for the GPT-2 family only"
    bb.inner.h = nn.ModuleList(list(bb.inner.h)[:n]); bb.inner.config.n_layer = n; return bb

def refresh_models(verbose=True):
    """re-scan /kaggle/input for Phase 1 models (for example after you attached another dataset to the running session)"""
    new = {k: v for k, v in discover_models().items() if k not in MODEL_PATHS}; MODEL_PATHS.update(new)
    if verbose and new: print("newly found Phase 1 models:", new)
    return sorted(MODEL_PATHS)


In [ ]:
# ================= CELL 5 — adaptation techniques: full, partial (layer-wise freezing), LoRA, rsLoRA, LoRMA+, LoRMA-pi, IA3 =================
from transformers.pytorch_utils import Conv1D
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale
class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B = 0 at init  (= W0 x + (alpha/r) B A (W0 x))
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale
class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0; A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A); i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)
class _Wrap:
    """expose the wrapped layer's weight / bias: some Hugging Face modules read them directly (T5's feed-forward dtype check does)"""
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
def proj_dims(m):
    return (m.in_features, m.out_features) if isinstance(m, nn.Linear) else tuple(m.weight.shape)      # HF Conv1D weight is [in, out]
class FusedQV(_Wrap, nn.Module):                  # wraps a FUSED q|k|v projection (GPT-2 Conv1D c_attn, ModernBERT Linear Wqkv); LoRA / LoRMA adapt q and v, IA3 rescales k and v
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); nx, nf = proj_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora": self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi": self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3": self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora": q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else: k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)
class IA3Input(_Wrap, nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base): super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(proj_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)
class LinAdapt(_Wrap, nn.Module):                 # the same adapters for separate nn.Linear projections (T5's q, k, v)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)
def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"
def freeze_lower(model, frac=0.5):
    """partial / layer-wise fine-tuning: freeze the embeddings and the lowest `frac` of the layers, train the rest"""
    inner, fam = model.backbone.inner, model.backbone.family
    for p in inner.parameters(): p.requires_grad = True
    if fam == "bilstm":
        n = inner.layers; k = int(n * frac)
        for p in inner.emb.parameters(): p.requires_grad = False
        for name, p in list(inner.fwd.named_parameters()) + list(inner.bwd.named_parameters()):
            if int(name.split("_l")[-1]) < k: p.requires_grad = False
        return
    if fam == "gpt2": layers, pre = list(inner.h), [inner.wte, inner.wpe]
    elif fam == "modernbert": layers, pre = list(inner.layers), [inner.embeddings]
    elif fam == "t5": layers, pre = list(inner.block), [inner.embed_tokens]
    else: raise ValueError(fam)
    k = int(len(layers) * frac)
    for mod in pre:
        for p in mod.parameters(): p.requires_grad = False
    for lay in layers[:k]:
        for p in lay.parameters(): p.requires_grad = False
METHODS = ("full", "partial", "lora", "rslora", "lorma_plus", "lorma_pi", "ia3")
ADAPTER_LR = {"lora": 5e-4, "rslora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 2e-4, "ia3": 3e-3}        # starting points; use lr_pilot(..., method=...) to check them
def apply_method(model, method, hp=None):
    assert method in METHODS, method; hp = hp or {}
    if method == "full":
        for p in model.backbone.parameters(): p.requires_grad = True
        return model
    if method == "partial": freeze_lower(model, hp.get("frac", 0.5)); return model
    fam, inner = model.backbone.family, model.backbone.inner
    for p in model.backbone.parameters(): p.requires_grad = False
    kind = {"lora": "lora", "rslora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]; kw = dict(r=hp.get("r", 16), alpha=hp.get("alpha", 32), drop=hp.get("adapter_dropout", 0.05), rs=(method == "rslora"))
    if fam == "gpt2":
        for blk in inner.h:
            assert isinstance(blk.attn.c_attn, Conv1D), "unexpected c_attn type in this transformers version"; blk.attn.c_attn = FusedQV(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    elif fam == "modernbert":
        for lay in inner.layers:
            assert isinstance(lay.attn.Wqkv, nn.Linear); lay.attn.Wqkv = FusedQV(lay.attn.Wqkv, kind, **kw)
            if kind == "ia3": lay.mlp.Wo = IA3Input(lay.mlp.Wo)
    elif fam == "t5":
        wrap_linear_targets(inner, kind, ("k", "v") if kind == "ia3" else ("q", "v"), **({} if kind == "ia3" else kw))
        if kind == "ia3":
            for _, parent in list(inner.named_modules()):
                if "DenseGatedActDense" in type(parent).__name__ or "DenseReluDense" in type(parent).__name__: parent.wo = IA3Input(parent.wo)
    else: raise NotImplementedError(f"'{method}' needs attention/linear layers: the {fam} model supports only 'full' and 'partial'")
    return model


In [ ]:
# ================= CELL 6 — trainer: any adaptation method, optional distillation, multi-window fine-tuning, EMA, multi-window inference, rate-matched threshold, submissions =================
import contextlib
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
def amp_ctx():
    if not AMP: return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)
DEFAULT_LR = {"gpt2": 3e-5, "modernbert": 5e-5, "t5": 5e-5, "bilstm": 5e-4}      # gpt2: best of 3e-5 / 1e-4 / 3e-4 in your pilot (the lowest won); the others are untested starting points: use lr_pilot() below
POLICY = "rate_matched"       # test-set decision rule: "rate_matched" (flag the top share of cases as 1, share = the dev positive rate; uses NO test labels) or "dev" (the threshold tuned on dev)
AGG_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]      # recency decay for averaging window scores: weight of window k is a**k (a=0: last window only, a=1: plain mean)

def prf(y, pred):
    f1s, ps, rs = [], [], []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0; f = 2 * p * r / (p + r) if p + r else 0.0; ps.append(p); rs.append(r); f1s.append(f)
    return dict(macro_f1=float(np.mean(f1s)), macro_p=float(np.mean(ps)), macro_r=float(np.mean(rs)), f1_0=f1s[0], f1_1=f1s[1], p1=ps[1], r1=rs[1])
def best_threshold(y, prob):
    best_f, best_t = -1.0, 0.5
    for t in sorted(np.arange(0.02, 0.981, 0.01), key=lambda t: abs(t - 0.5)):
        f = prf(y, (prob >= t).astype(int))["macro_f1"]
        if f > best_f + 1e-12: best_f, best_t = f, float(t)
    return best_t, best_f
def roc_auc(y, s):
    r = pd.Series(s).rank().values; n1 = int((y == 1).sum()); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def rm_predict(score, prior):
    """rate matching: the top round(prior*N) scores become class 1. Needs only the scores and the class prior (from dev), never labels of the set being predicted."""
    k = int(round(prior * len(score))); pred = np.zeros(len(score), dtype=int); pred[np.argsort(-score, kind="stable")[:k]] = 1; return pred

@torch.no_grad()
def predict_windows(model, tails, L, W, tok_budget=16384):
    """margin (logit1 - logit0) of each of the last W windows of every case; NaN where the case has fewer windows"""
    model.eval(); n = len(tails); nw = np.array([n_windows(len(t), L, W) for t in tails]); Z = np.full((n, W), np.nan, dtype=np.float32); mb = max(1, tok_budget // L)
    for k in range(W):
        idx_k = np.where(nw > k)[0]
        if len(idx_k) == 0: continue
        order = idx_k[np.argsort([-len(tails[i]) for i in idx_k], kind="stable")]
        for s in range(0, len(order), mb):
            sel = order[s:s + mb]; ids, mask = make_window_batch(tails, sel, [k] * len(sel), L)
            with amp_ctx(): lg = model(ids.to(device), mask.to(device)).float()
            Z[sel, k] = (lg[:, 1] - lg[:, 0]).cpu().numpy()
    return Z
def aggregate(Z, a):
    w = np.array([a ** k for k in range(Z.shape[1])], dtype=np.float32); den = np.where(np.isnan(Z), 0.0, w[None, :]).sum(1)
    return np.nansum(Z * w[None, :], axis=1) / den

class EMA:
    """exponential moving average of the weights; evaluated instead of the raw weights (smoother, usually better at the end of fine-tuning)"""
    def __init__(self, model, decay):
        self.params = [p for p in model.parameters() if p.requires_grad]; self.shadow = [p.detach().clone() for p in self.params]; self.decay = decay; self.backup = None
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step)); torch._foreach_mul_(self.shadow, d); torch._foreach_add_(self.shadow, [p.detach() for p in self.params], alpha=1 - d)
    @torch.no_grad()
    def apply(self):
        self.backup = [p.detach().clone() for p in self.params]
        for p, s in zip(self.params, self.shadow): p.copy_(s)
    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup): p.copy_(b)
        self.backup = None

def to_py(x): return x.item() if hasattr(x, "item") else x
def write_submission(path, rids, preds): json.dump([{"id": to_py(r), "pred": int(p)} for r, p in zip(rids, preds)], open(path, "w"))
def validate_submission(path, rids):
    sub = json.load(open(path)); assert isinstance(sub, list) and len(sub) == len(rids), f"{path}: expected {len(rids)} predictions, found {len(sub)}"
    for s, r in zip(sub, rids):
        assert set(s) == {"id", "pred"}, f"bad keys {set(s)}"; assert type(s["pred"]) is int and s["pred"] in (0, 1), f"pred must be the integer 0 or 1, got {s['pred']!r}"; assert s["id"] == to_py(r), f"id mismatch {s['id']!r} vs {to_py(r)!r}"
    assert len({str(s['id']) for s in sub}) == len(sub), "duplicate ids"; return True

def run_experiment(name, model_key, method="full", hp=None, n_layers=None, kd=None, L=512, W=4, epochs=3.0, max_minutes=None, lr=None, head_lr=1e-3, bs=16, wd=0.01, warmup=0.06, p_tail=0.5, ema_decay=0.999,
                   backbone_dropout=0.0, pool="last", dropout=0.1, seed=SEED, max_train_docs=None, dev_cap=None, test_cap=None, evals=4, write=True, policy=None, save_model=False, control=False):
    """method: full | partial | lora | rslora | lorma_plus | lorma_pi | ia3 ;  kd=dict(path=..., T=3.0, alpha=0.5) distils from the teacher margins made by make_teacher() ;  n_layers=6 builds a half-depth GPT-2 student ;  control=True marks an ablation run that is never one of the six submissions"""
    t_start = time.time(); seed_all(seed); rng = np.random.default_rng(seed); policy = policy or POLICY
    assert model_key in MODEL_PATHS, f"'{model_key}' is not available: pre-train it first (found: {list(MODEL_PATHS)})"
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[model_key]); L = min(L, ctx)
    if n_layers: truncate_layers(bb, n_layers)
    lr = lr or (DEFAULT_LR[fam] if method in ("full", "partial") else ADAPTER_LR[method]); technique = "kd" if kd else method
    nd = set_backbone_dropout(bb, backbone_dropout) if backbone_dropout else 0
    model = DocClassifier(bb, hidden, causal, 1, dropout, pool); apply_method(model, method, hp); model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tr, dv, te = DATA["train"], DATA["dev"], DATA["test"]
    tr_idx = np.arange(len(tr["tails"])) if not max_train_docs else rng.choice(len(tr["tails"]), min(max_train_docs, len(tr["tails"])), replace=False)
    dv_idx = np.arange(len(dv["tails"])) if not dev_cap else rng.choice(len(dv["tails"]), min(dev_cap, len(dv["tails"])), replace=False); te_idx = np.arange(len(te["tails"])) if not test_cap else np.arange(min(test_cap, len(te["tails"])))
    dv_tails = [dv["tails"][i] for i in dv_idx]; y_dev = dv["labels"][dv_idx]; te_tails = [te["tails"][i] for i in te_idx]; prior = float(np.mean(dv["labels"] == 1))
    sub_n = min(1500, len(dv_idx)); sub = rng.choice(len(dv_idx), sub_n, replace=False); sub_tails = [dv_tails[i] for i in sub]; y_sub = y_dev[sub]
    nwin = np.array([n_windows(len(t), L, W) for t in tr["tails"]])
    cnt = np.bincount(tr["labels"][tr_idx], minlength=2).astype(float); cw = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    mb = max(1, 8192 // L); accum = max(1, math.ceil(bs / mb)); docs_per_step = mb * accum
    bb_train = [p for p in model.backbone.parameters() if p.requires_grad]; groups = ([{"params": bb_train, "lr": lr}] if bb_train else []) + [{"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": head_lr}]
    tm = None
    if kd:
        tm = np.load(kd["path"]).astype(np.float32); assert len(tm) == len(tr["tails"]), "teacher margins do not match the training set: run make_teacher() on the full training set"
    opt = torch.optim.AdamW(groups, weight_decay=wd); base_lrs = [g["lr"] for g in groups]; scaler = torch.amp.GradScaler("cuda", enabled=AMP and not USE_BF16); ema = EMA(model, ema_decay)
    def sample_ks(idx):
        ks = np.zeros(len(idx), dtype=int)
        for j, i in enumerate(idx):
            if nwin[i] > 1 and rng.random() > p_tail: ks[j] = rng.integers(1, nwin[i])      # an earlier window, with probability 1 - p_tail
        return ks
    def micro(ids_idx):
        ids, mask = make_window_batch(tr["tails"], ids_idx, sample_ks(ids_idx), L); y = torch.from_numpy(tr["labels"][ids_idx]).to(device)
        with amp_ctx(): logits = model(ids.to(device), mask.to(device))
        logits = logits.float(); ce = F.cross_entropy(logits, y, weight=cw)
        if tm is None: return ce
        T_, a_ = kd.get("T", 3.0), kd.get("alpha", 0.5); pt = torch.sigmoid(torch.from_numpy(tm[ids_idx]).to(device) / T_); td = torch.stack([1 - pt, pt], 1)      # teacher P(class 1), softened by the temperature
        kl = (td * (torch.log(td.clamp_min(1e-8)) - F.log_softmax(logits / T_, -1))).sum(1).mean() * T_ * T_
        return a_ * ce + (1 - a_) * kl
    model.train(); times = []
    for j in range(4):                                                            # throughput probe (forward + backward only, no optimizer step); the first is warm-up
        t0 = time.time(); loss = micro(rng.choice(tr_idx, mb)); scaler.scale(loss).backward(); opt.zero_grad(set_to_none=True)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    micro_s = float(np.mean(times[1:])); sec_step = micro_s * accum; per_doc_fwd = micro_s / mb / 3.0
    steps = max(10, int(math.ceil(epochs * len(tr_idx) / docs_per_step))); eval_s = ((len(dv_tails) + len(te_tails)) * W + evals * sub_n) * per_doc_fwd
    if max_minutes: steps = min(steps, max(10, int((max_minutes * 60 * 0.92 - eval_s) / sec_step)))
    print(f"[{name}] {fam} | technique {technique}{' (method ' + method + ')' if kd else ''} | L={L} W={W} p_tail={p_tail} | {n_params/1e6:.0f}M params, {n_train/1e6:.2f}M trained ({n_train/n_params:.1%}) | lr {lr:g} (head {head_lr:g}) | EMA {ema_decay} | {docs_per_step} cases/step\n"
          f"   plan: {steps} steps = {steps*docs_per_step} cases = {steps*docs_per_step/len(tr_idx):.2f} epochs | {sec_step:.2f}s/step -> ~{steps*sec_step/60:.0f} min training + ~{eval_s/60:.1f} min inference")
    perm, ptr, evals_at, run_loss, hist = rng.permutation(tr_idx), 0, {int(round(steps * (i + 1) / evals)) for i in range(evals)}, 0.0, []
    for step in range(1, steps + 1):
        w = min(step / max(1, int(warmup * steps)), (steps - step + 1) / max(1, steps - int(warmup * steps)), 1.0)
        for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * w
        for _ in range(accum):
            if ptr + mb > len(perm): perm, ptr = rng.permutation(tr_idx), 0
            loss = micro(perm[ptr:ptr + mb]); ptr += mb; scaler.scale(loss / accum).backward(); run_loss += loss.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(step)
        if step % max(1, steps // 12) == 0: print(f"   step {step}/{steps} | train loss {run_loss/max(1, steps//12):.4f} | {(time.time()-t_start)/60:.1f} min", flush=True); run_loss = 0.0
        if step in evals_at:                                                       # monitoring only: EMA weights, last window, dev subset (no snapshot selection: the final EMA weights are used)
            ema.apply(); z = predict_windows(model, sub_tails, L, 1)[:, 0]; ema.restore(); model.train(); a_ = roc_auc(y_sub, z); t_, f_ = best_threshold(y_sub, sigmoid(z)); hist.append((step, round(a_, 4), round(f_, 4)))
            print(f"   [dev subset {sub_n}, EMA, last window] step {step}: AUC {a_:.4f} | macro-F1 {f_:.4f}", flush=True)
    ema.apply()                                                                    # final model = EMA weights
    Zd, Zt = predict_windows(model, dv_tails, L, W), predict_windows(model, te_tails, L, W)
    aucs = {a: roc_auc(y_dev, aggregate(Zd, a)) for a in AGG_GRID}; a_best = max(AGG_GRID, key=lambda a: (round(aucs[a], 6), -a)); m_dev, m_te = aggregate(Zd, a_best), aggregate(Zt, a_best); p_dev, p_te = sigmoid(m_dev), sigmoid(m_te)
    thr, f_tuned = best_threshold(y_dev, p_dev); at05 = prf(y_dev, (p_dev >= 0.5).astype(int)); rm = prf(y_dev, rm_predict(p_dev, prior)); tuned = prf(y_dev, (p_dev >= thr).astype(int))
    pred_te = rm_predict(p_te, prior) if policy == "rate_matched" else (p_te >= thr).astype(int)
    res = dict(name=name, model=model_key, family=fam, method=method, technique=technique, kd=bool(kd), n_layers=n_layers or 0, control=bool(control), L=L, W=W, p_tail=p_tail, pool=pool, params_M=round(n_params / 1e6, 1), trainable_M=round(n_train / 1e6, 3), lr=lr, epochs=round(steps * docs_per_step / len(tr_idx), 2), steps=steps, agg_a=a_best,
               dev_auc=aucs[a_best], dev_auc_last_window=aucs[0.0], dev_f1_ratematched=rm["macro_f1"], dev_f1_tuned=f_tuned, dev_f1_at_0p5=at05["macro_f1"], threshold=thr, dev_p1=tuned["p1"], dev_r1=tuned["r1"], dev_cases=len(y_dev),
               test_pos_rate_at_dev_thr=float(np.mean(p_te >= thr)), policy=policy, minutes=round((time.time() - t_start) / 60, 1), history=str(hist))
    print(f"[{name}] DEV: AUC {aucs[a_best]:.4f} (last window only: {aucs[0.0]:.4f}; chosen recency decay a={a_best}) | macro-F1 rate-matched {rm['macro_f1']:.4f} | tuned {f_tuned:.4f} (thr {thr:.2f}) | at 0.5 {at05['macro_f1']:.4f} | {res['minutes']} min\n"
          f"   test: share predicted positive at the dev threshold = {res['test_pos_rate_at_dev_thr']:.3f} (dev positive rate {prior:.3f}) -> policy '{policy}' used for the submission file")
    if write:
        rd = WORK / "runs" / name; rd.mkdir(exist_ok=True); np.save(rd / "dev_Z.npy", Zd); np.save(rd / "test_Z.npy", Zt); np.save(rd / "dev_probs.npy", p_dev); np.save(rd / "test_probs.npy", p_te); json.dump(res, open(rd / "result.json", "w"), indent=1)
        if save_model: torch.save({"state": model.state_dict()}, rd / "model.pt"); json.dump(dict(model_key=model_key, method=method, hp=hp, n_layers=n_layers, pool=pool, L=L, W=W, agg_a=a_best), open(rd / "meta.json", "w"))
        rids = [te["rids"][i] for i in te_idx]; write_submission(SUBDIR / f"{name}.json", rids, pred_te); validate_submission(SUBDIR / f"{name}.json", rids)
        row = pd.DataFrame([res]); led = pd.concat([pd.read_csv(LEDGER).query('name != @name'), row]) if LEDGER.exists() else row; led.to_csv(LEDGER, index=False)
        print(f"   saved {SUBDIR / (name + '.json')} ({len(rids)} predictions, format validated) and the ledger row")
    del model, opt, bb, ema; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res

def lr_pilot(model_key, grid=(1.5e-5, 3e-5, 1e-4), minutes=10, L=512, **kw):
    """short runs on a subset; picks the learning rate by dev AUC (threshold-free, less noisy than F1). Differences below about 0.01 AUC are noise: prefer the middle value then."""
    out = {}
    for lr_ in grid:
        r = run_experiment(f"pilot_{model_key}_{lr_:g}", model_key, L=L, W=1, epochs=99, max_minutes=minutes, lr=lr_, max_train_docs=10000, dev_cap=1500, test_cap=50, evals=1, write=False, ema_decay=0.99, **kw); out[lr_] = r["dev_auc"]
    best = max(out, key=out.get); print("\nLR pilot (dev AUC):", {f"{k:g}": round(v, 4) for k, v in out.items()}, "-> best", f"{best:g}"); return best

def load_finetuned(run_name):
    """rebuild a fine-tuned model from runs/<name>/model.pt + meta.json (written when save_model=True)"""
    rd = WORK / "runs" / run_name; meta = json.load(open(rd / "meta.json")); ck = torch.load(rd / "model.pt", map_location="cpu", weights_only=False)
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[meta["model_key"]])
    if meta.get("n_layers"): truncate_layers(bb, meta["n_layers"])
    model = DocClassifier(bb, hidden, causal, 1, 0.0, meta.get("pool", "last")); apply_method(model, meta["method"], meta.get("hp")); model.load_state_dict(ck["state"]); return model.to(device).eval(), meta

@torch.no_grad()
def make_teacher(run_names, out=None):
    """teacher for distillation: the mean margin (logit1 - logit0, all windows, the run's own recency decay) of finished runs on EVERY training case. Costs one inference pass over the training set per run (about 15-30 min)."""
    tr = DATA["train"]; margins = []
    for name in run_names:
        model, meta = load_finetuned(name); t0 = time.time(); Z = predict_windows(model, tr["tails"], meta["L"], meta["W"]); m = aggregate(Z, meta["agg_a"]); margins.append(m)
        print(f"teacher {name}: training-set accuracy {((m > 0) == (tr['labels'] == 1)).mean():.3f} (high is expected: it was trained on these cases) | {(time.time()-t0)/60:.1f} min")
        del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    tm = np.mean(margins, 0).astype(np.float32); path = Path(out) if out else WORK / "teacher_margin.npy"; np.save(path, tm)
    print(f"teacher = mean margin of {run_names} -> {path} | margin std {tm.std():.2f} (distillation temperature T softens it)"); return str(path)

# ---- run only what can run: experiments whose Phase 1 model is missing are skipped (with a message), everything else goes on ----
def run_or_skip(name, model_key, **kw):
    """run an experiment if its Phase 1 model is available (re-scanning /kaggle/input first); otherwise print why and carry on"""
    if model_key not in MODEL_PATHS: refresh_models(verbose=False)
    if model_key not in MODEL_PATHS:
        print(f"SKIPPED {name}: the Phase 1 '{model_key}' model is not available yet (found: {sorted(MODEL_PATHS)}). Attach it or finish its pre-training, then re-run this cell."); return None
    return run_experiment(name, model_key, **kw)
def pilot_or_skip(model_key, default_lr, **kw):
    if model_key not in MODEL_PATHS: refresh_models(verbose=False)
    if model_key not in MODEL_PATHS:
        print(f"SKIPPED the learning-rate pilot for '{model_key}' (model not available); the default {default_lr:g} is used if you run its experiment later."); return default_lr
    return lr_pilot(model_key, **kw)
def show_plan():
    plan = [("P1_gpt2_full", "gpt2_new", "full fine-tuning"), ("P2_enc_full", "modernbert", "full fine-tuning"), ("P3_enc_lorma", "modernbert", "LoRMA+ (compulsory)"), ("P4_t5_lora", "t5", "LoRA"),
            ("P5_bilstm_partial", "bilstm", "partial fine-tuning"), ("P6_gpt2_ia3", "gpt2_new", "IA3 (optional)"), ("P7_gpt2_6L_kd", "gpt2_new", "knowledge distillation (needs a saved teacher from P1 and/or P2)")]
    refresh_models(verbose=False); done = set(pd.read_csv(LEDGER).name) if LEDGER.exists() else set()
    print("Phase 1 models available:", sorted(MODEL_PATHS), "| MISSING:", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS] or "none")
    print(f"{'run':20s} {'model':11s} {'technique':62s} status")
    for n, m, tech in plan:
        teach = (WORK / "runs" / "P1_gpt2_full" / "model.pt").exists() or (WORK / "runs" / "P2_enc_full" / "model.pt").exists()
        st = "done" if n in done else ("SKIPPED: model missing" if m not in MODEL_PATHS else ("will run once P1 or P2 has finished" if (n == "P7_gpt2_6L_kd" and not teach) else "will run"))
        print(f"{n:20s} {m:11s} {tech:62s} {st}")


In [ ]:
# ================= CELL 7 — SMOKE TEST (~4 min): windows + EMA + rate matching + one adapter method on the first available model; writes nothing. Must pass before the real runs. =================
refresh_models(verbose=False); SMOKE_KEY = next((k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k in MODEL_PATHS), None)
assert SMOKE_KEY, "no Phase 1 model found under /kaggle/input: attach at least one dataset that contains a final_model.pt"
SMOKE_METHOD = "partial" if SMOKE_KEY == "bilstm" else "lorma_plus"          # the BiLSTM has no attention layers, so it cannot take adapters
res = run_experiment("smoke", SMOKE_KEY, method=SMOKE_METHOD, hp=dict(r=16, alpha=32), L=512, W=2, epochs=99, max_minutes=2.5, max_train_docs=600, dev_cap=300, test_cap=40, evals=1, write=False, ema_decay=0.98)
assert np.isfinite(res["dev_auc"]); print(f"\nsmoke test OK ({SMOKE_METHOD} on {SMOKE_KEY}). Expect a LOW score here (about 600 training cases); the plan line above shows the real speed, so you can size the experiments below.")


In [ ]:
# ================= CELL 8 — THE EXPERIMENTS: the techniques the rules ask for, across the Phase 1 architectures (the old 3,000-step GPT-2 is NOT used) =================
# Experiments whose Phase 1 model is missing are SKIPPED with a message; the rest run. Attach a missing model (or finish its pre-training) and re-run just that cell: models are re-scanned automatically.
#  full fine-tuning   GPT-2 (P1) | encoder (P2)         LoRMA+ (COMPULSORY) encoder (P3)       LoRA T5 encoder (P4)       partial fine-tuning BiLSTM (P5)
#  IA3 GPT-2 (P6, optional)                              knowledge distillation: half-depth GPT-2 student of the P1 + P2 ensemble (P7), control P7c without distillation
# Differences below about 0.01 AUC (about 0.02 macro-F1) are within noise on 3,505 dev cases.
show_plan()


In [ ]:
# P1: GPT-2, FULL fine-tuning, multi-window, 3 epochs, EMA (saved as a distillation teacher)   (~80 min on a T4)
P1 = run_or_skip("P1_gpt2_full", "gpt2_new", method="full", L=512, W=4, epochs=3, lr=3e-5, p_tail=0.5, save_model=True)


In [ ]:
# P2: ENCODER, FULL fine-tuning (saved as a distillation teacher). Learning rate: pilot first (~30 min), then the run (~70 min). Skipped if the encoder is missing.
LR_ENC = pilot_or_skip("modernbert", 5e-5, grid=(2e-5, 5e-5, 1.5e-4), minutes=10)
P2 = run_or_skip("P2_enc_full", "modernbert", method="full", L=512, W=4, epochs=3, lr=LR_ENC, p_tail=0.5, save_model=True)


In [ ]:
# P3: ENCODER, LoRMA+ (COMPULSORY technique): multiplicative low-rank adaptation of q and v, backbone frozen. Adapters converge more slowly, so 4 epochs.   (pilot ~30 min, run ~60 min). Skipped if the encoder is missing.
HP_LORMA = dict(r=16, alpha=32)
LR_LORMA = pilot_or_skip("modernbert", ADAPTER_LR["lorma_plus"], grid=(2e-4, 5e-4, 1.5e-3), minutes=10, method="lorma_plus", hp=HP_LORMA)
P3 = run_or_skip("P3_enc_lorma", "modernbert", method="lorma_plus", hp=HP_LORMA, L=512, W=4, epochs=4, lr=LR_LORMA, p_tail=0.5)
# optional variant for the report (LoRMA-pi, permutation-based rank inflation):  run_or_skip("P3b_enc_lorma_pi", "modernbert", method="lorma_pi", hp=HP_LORMA, L=512, W=4, epochs=4, lr=2e-4, p_tail=0.5)


In [ ]:
# P4: T5 ENCODER, LoRA (additive low-rank adaptation of q and v, backbone frozen)   (pilot ~30 min, run ~60 min). Skipped if the T5 model is missing.
HP_LORA = dict(r=16, alpha=32)
LR_LORA = pilot_or_skip("t5", ADAPTER_LR["lora"], grid=(2e-4, 5e-4, 1.5e-3), minutes=10, method="lora", hp=HP_LORA)
P4 = run_or_skip("P4_t5_lora", "t5", method="lora", hp=HP_LORA, L=512, W=4, epochs=4, lr=LR_LORA, p_tail=0.5)


In [ ]:
# P5: BiLSTM, PARTIAL (layer-wise) fine-tuning: embeddings and the lowest LSTM layer frozen, the rest trained. Longer windows are cheap for a recurrent model.   (pilot ~30 min, run ~60 min)
# Skipped while the BiLSTM is not pre-trained / attached: just re-run this cell later.
LR_RNN = pilot_or_skip("bilstm", DEFAULT_LR["bilstm"], grid=(2e-4, 5e-4, 1e-3), minutes=10, L=1024, method="partial", hp=dict(frac=0.5))
P5 = run_or_skip("P5_bilstm_partial", "bilstm", method="partial", hp=dict(frac=0.5), L=1024, W=4, epochs=3, lr=LR_RNN, p_tail=0.5)


In [ ]:
# P6 (optional): GPT-2, IA3 (learned rescaling vectors; the smallest adapter)   (pilot ~30 min, run ~60 min)
LR_IA3 = pilot_or_skip("gpt2_new", ADAPTER_LR["ia3"], grid=(1e-3, 3e-3, 1e-2), minutes=10, method="ia3")
P6 = run_or_skip("P6_gpt2_ia3", "gpt2_new", method="ia3", L=512, W=4, epochs=4, lr=LR_IA3, p_tail=0.5)


In [ ]:
# P7: KNOWLEDGE DISTILLATION. Teacher = mean margin of the saved P1 (GPT-2) and/or P2 (encoder) models on the training cases (one inference pass each, ~15 min per model).
# Student = the first 6 layers of the Phase 1 GPT-2. Loss = alpha * hard-label cross-entropy + (1 - alpha) * T^2 * KL(teacher || student), on the sampled window of each case.
# If only one teacher run exists it is used alone; if none exists the cell is skipped (run P1 and/or P2 first: they are saved automatically).
TEACH = [n for n in ("P1_gpt2_full", "P2_enc_full") if (WORK / "runs" / n / "model.pt").exists()]
if not TEACH: print("SKIPPED P7 (distillation): no saved teacher model yet: run P1 and/or P2 first.")
elif "gpt2_new" not in MODEL_PATHS and "gpt2_new" not in refresh_models(verbose=False): print("SKIPPED P7 (distillation): the student is a half-depth GPT-2 and the GPT-2 Phase 1 model is not available.")
else:
    print("teacher runs:", TEACH, "(single teacher)" if len(TEACH) == 1 else "(ensemble)"); TEACHER = make_teacher(TEACH)
    P7 = run_or_skip("P7_gpt2_6L_kd", "gpt2_new", method="full", n_layers=6, kd=dict(path=TEACHER, T=3.0, alpha=0.5), L=512, W=4, epochs=3, lr=5e-5, p_tail=0.5)                # ~50 min
    # CONTROL (same student, hard labels only) so the benefit of distillation is measured; it is never one of the six submissions:
    P7c = run_or_skip("P7c_gpt2_6L_nokd", "gpt2_new", method="full", n_layers=6, L=512, W=4, epochs=3, lr=5e-5, p_tail=0.5, control=True)                                    # ~50 min


In [ ]:
# OPTIONAL extra runs for the ensemble / a stronger sixth experiment (each ~60-80 min). The selection cell chooses the best valid six from everything in the ledger.
# P8 = run_experiment("P8_t5_full", "t5", method="full", L=512, W=4, epochs=3, lr=5e-5, p_tail=0.5)
# P9 = run_experiment("P9_bilstm_full", "bilstm", method="full", L=1024, W=4, epochs=3, lr=5e-4, p_tail=0.5)
# P10 = run_experiment("P10_gpt2_full_seed2", "gpt2_new", method="full", L=512, W=4, epochs=3, lr=3e-5, p_tail=0.5, seed=7)


In [ ]:
# ================= CELL 9 — pick the six submissions THAT SATISFY THE RULES, build the ensemble, write prediction.json, validate, package =================
import shutil
from itertools import combinations
def md_table(df):
    try: return df.to_markdown(index=False)
    except Exception: return "```\n" + df.to_string(index=False) + "\n```"
assert LEDGER.exists(), "no experiment has finished yet: run at least one experiment cell (cell 8) first"
led = pd.read_csv(LEDGER); led = led[~led.name.str.startswith("pilot")].sort_values("dev_auc", ascending=False).reset_index(drop=True); led["control"] = led["control"].astype(str).str.lower().eq("true")
cols = ["name", "model", "technique", "trainable_M", "epochs", "agg_a", "dev_auc", "dev_auc_last_window", "dev_f1_ratematched", "dev_f1_tuned", "test_pos_rate_at_dev_thr", "minutes"]; print(led[cols].round(4).to_string(index=False))
rids = DATA["test"]["rids"]; y_dev = DATA["dev"]["labels"]; prior = float(np.mean(y_dev == 1))
# ---- the rules: all four Phase 1 architectures, compulsory LoRMA, at least one distillation run, at least five different technique families, six experiments ----
FAMILY = {"full": "full fine-tuning", "partial": "partial fine-tuning", "lora": "LoRA", "rslora": "LoRA", "lorma_plus": "LoRMA", "lorma_pi": "LoRMA", "ia3": "IA3", "kd": "knowledge distillation"}
REQ_MODELS = ("gpt2_new", "modernbert", "t5", "bilstm")
def rule_check(sub):
    fam = {FAMILY[t] for t in sub.technique}
    return dict(all_four_architectures=set(REQ_MODELS) <= set(sub.model), has_lorma="LoRMA" in fam, has_distillation="knowledge distillation" in fam, five_techniques=len(fam) >= 5, families=sorted(fam))
def choose_six(cand):
    """the six runs with the highest mean dev AUC among all combinations that satisfy every rule (controls excluded)"""
    best = None
    for combo in combinations(range(len(cand)), 6):
        sub = cand.iloc[list(combo)]; rc = rule_check(sub)
        if rc["all_four_architectures"] and rc["has_lorma"] and rc["has_distillation"] and rc["five_techniques"]:
            sc = float(sub.dev_auc.mean())
            if best is None or sc > best[0]: best = (sc, list(sub.name))
    return best
cand = led[~led.control].reset_index(drop=True); assert len(cand) >= 1, "no experiments in the ledger yet: run at least one in cell 8"
def choose_set(cand, size=6):
    """the best set of up to six runs: fully rule-compliant sets always win; otherwise the set that satisfies the most rules (architectures covered, LoRMA, distillation, technique families), then the highest mean dev AUC"""
    n = min(size, len(cand)); best = None
    for combo in combinations(range(len(cand)), n):
        sub = cand.iloc[list(combo)]; rc = rule_check(sub)
        strict = bool(n == 6 and rc["all_four_architectures"] and rc["has_lorma"] and rc["has_distillation"] and rc["five_techniques"])
        key = (strict, len(set(sub.model) & set(REQ_MODELS)), rc["has_lorma"], rc["has_distillation"], min(len(rc["families"]), 5), float(sub.dev_auc.mean()))
        if best is None or key > best[0]: best = (key, list(sub.name))
    return best
(key_, FINAL) = choose_set(cand); sub = led.set_index("name").loc[FINAL].reset_index(); rc = rule_check(sub); strict = bool(len(FINAL) == 6 and rc["all_four_architectures"] and rc["has_lorma"] and rc["has_distillation"] and rc["five_techniques"])
unmet = ([] if len(FINAL) == 6 else [f"only {len(FINAL)} experiment(s) so far (six needed)"]) + ([] if rc["all_four_architectures"] else [f"architectures not covered yet: {sorted(set(REQ_MODELS) - set(sub.model))}"]) + ([] if rc["has_lorma"] else ["no LoRMA run (P3)"]) \
        + ([] if rc["has_distillation"] else ["no distillation run (P7)"]) + ([] if rc["five_techniques"] else [f"only {len(rc['families'])} technique families (five needed)"])
status = "FULLY COMPLIANT with the rules" if strict else "NOT FULLY COMPLIANT YET: " + "; ".join(unmet)
print(f"\nSELECTED SET ({len(FINAL)} experiment(s), mean dev AUC {float(sub.dev_auc.mean()):.4f}): {status}"); print(sub[["name", "model", "technique", "dev_auc", "dev_f1_ratematched"]].round(4).to_string(index=False)); print("rule check:", rc)
if not strict: print("-> prediction.json below is still produced from what exists. Finish the missing models/experiments in cell 8 and re-run this cell: it will then pick a compliant six automatically.")
FINAL_DIR = WORK / "final_six"; shutil.rmtree(FINAL_DIR, ignore_errors=True); FINAL_DIR.mkdir()
for k, n in enumerate(FINAL, 1): validate_submission(SUBDIR / f"{n}.json", rids); (FINAL_DIR / f"submission_{k}_{n}.json").write_text((SUBDIR / f"{n}.json").read_text())
ctl = led[led.control]
(FINAL_DIR / "requirements_report.md").write_text("# Phase 4 requirement check\n\n" + md_table(sub[["name", "model", "technique", "trainable_M", "dev_auc", "dev_f1_ratematched", "dev_f1_tuned"]].round(4)) +
    f"\n\n**STATUS: {status}**\n\n- all four Phase 1 architectures: {rc['all_four_architectures']}\n- LoRMA included: {rc['has_lorma']}\n- knowledge distillation included: {rc['has_distillation']}\n- technique families ({len(rc['families'])}): {', '.join(rc['families'])}\n- the old 3,000-step GPT-2 is not used anywhere\n" +
    ("\n## Distillation control (same student, hard labels only)\n\n" + md_table(ctl[["name", "dev_auc", "dev_f1_ratematched"]].round(4)) + "\n" if len(ctl) else ""))
# ---- ensemble of everything non-control: average of dev-standardised margins; number of models chosen on dev AUC, preferring MORE models when within 0.002 ----
def margins(n):
    pd_, pt_ = np.load(WORK / "runs" / n / "dev_probs.npy"), np.load(WORK / "runs" / n / "test_probs.npy"); lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))); md, mt = lg(pd_), lg(pt_); s = md.std() + 1e-9
    return md / s, mt / s
M = {n: margins(n) for n in cand.name}; order = list(cand.sort_values("dev_auc", ascending=False).name); aucs_k = [roc_auc(y_dev, np.mean([M[n][0] for n in order[:k]], 0)) for k in range(1, len(order) + 1)]
best_auc = max(aucs_k); k_ens = max(k for k, a in enumerate(aucs_k, 1) if a >= best_auc - 0.002); ens_names = order[:k_ens]
z_dev = np.mean([M[n][0] for n in ens_names], 0); z_te = np.mean([M[n][1] for n in ens_names], 0)
print("\nensemble dev AUC by number of models (best first):", [round(a, 4) for a in aucs_k], f"\n-> using the top {k_ens}: {ens_names}")
e_rm = prf(y_dev, rm_predict(z_dev, prior)); print(f"ensemble DEV: AUC {roc_auc(y_dev, z_dev):.4f} | macro-F1 rate-matched {e_rm['macro_f1']:.4f} | best single run: AUC {cand.dev_auc.max():.4f}, rate-matched macro-F1 {cand.dev_f1_ratematched.max():.4f}")
write_submission(Path("/kaggle/working/prediction_ensemble.json"), rids, rm_predict(z_te, prior)); validate_submission(Path("/kaggle/working/prediction_ensemble.json"), rids)
best_single = cand.sort_values("dev_auc", ascending=False).iloc[0]["name"]; shutil.copy(SUBDIR / f"{best_single}.json", Path("/kaggle/working/prediction_best_single.json"))
use_ens = roc_auc(y_dev, z_dev) >= cand.dev_auc.max() + 0.002; shutil.copy("/kaggle/working/prediction_ensemble.json" if use_ens else str(SUBDIR / f"{best_single}.json"), "/kaggle/working/prediction.json"); validate_submission(Path("/kaggle/working/prediction.json"), rids)
print(f"\nprediction.json = {'the ensemble' if use_ens else best_single} (chosen on dev AUC only; the leaderboard plays no part) | positives predicted on test: {np.mean([p['pred'] for p in json.load(open('/kaggle/working/prediction.json'))]):.3f}")
with zipfile.ZipFile("/kaggle/working/cjpe_results_v3.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for n in led.name:
        for f in ("dev_Z.npy", "test_Z.npy", "dev_probs.npy", "test_probs.npy", "result.json"): z.write(WORK / "runs" / n / f, f"runs/{n}/{f}")
        z.write(SUBDIR / f"{n}.json", f"submissions/{n}.json")
    z.write(LEDGER, "ledger.csv")
    for p in sorted(FINAL_DIR.glob("*")): z.write(p, "final_six/" + p.name)
print("files in /kaggle/working: prediction.json, prediction_ensemble.json, prediction_best_single.json, final_six/ (six submissions + requirements_report.md), cjpe_results_v3.zip")
print("RULES OF THE ROAD: choose by DEV only; look at the leaderboard once per final candidate; differences under ~0.02 on 1,517 test cases are noise (one standard error is about 0.012).")


In [ ]:
# ================= CELL 10 — DIAGNOSTICS on dev for the ensemble above (where does it fail?) =================
pred_d = rm_predict(z_dev, prior); lens = DATA["dev"]["lens"]; qs = np.quantile(lens, [0, .25, .5, .75, 1.0]); rows = []
for lo, hi in zip(qs[:-1], qs[1:]):
    sel = (lens >= lo) & (lens <= hi); r = prf(y_dev[sel], pred_d[sel]); rows.append(dict(tokens=f"{int(lo)}-{int(hi)}", cases=int(sel.sum()), macro_f1=round(r["macro_f1"], 3), acc=round(float((pred_d[sel] == y_dev[sel]).mean()), 3), auc=round(roc_auc(y_dev[sel], z_dev[sel]), 3)))
print("dev performance by case length (token quartiles):"); print(pd.DataFrame(rows).to_string(index=False)); al = prf(y_dev, pred_d)
print(f"\noverall (rate-matched): macro-F1 {al['macro_f1']:.4f} | class-1 precision {al['p1']:.4f} recall {al['r1']:.4f} | class-0 F1 {al['f1_0']:.4f} class-1 F1 {al['f1_1']:.4f}")
conf = pd.crosstab(pd.Series(y_dev, name="true"), pd.Series(pred_d, name="predicted")); print("\nconfusion matrix (dev):"); print(conf.to_string())
print("\nrun-to-run agreement on dev (correlation of standardised margins) - low values = diverse models, good for the ensemble:"); print(pd.DataFrame({n: M[n][0] for n in order}).corr().round(2).to_string())
